In [34]:
import pandas as pd
import numpy as np


products = pd.read_csv('products.csv')
departments = pd.read_csv('departments.csv')
order_products_prior = pd.read_csv('order_products_prior_final.csv')
orders = pd.read_csv('orders_final.csv')


In [35]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [36]:
orders.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1,1,1,5,7,NaN
1,2,1,2,6,23,11.0
2,3,1,3,2,8,9.0
3,4,1,4,1,20,18.0
4,5,1,5,3,11,12.0


In [37]:
order_products_prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,1,31140,1,0
1,1,486,2,0
2,1,22325,3,0
3,1,12697,4,0
4,1,23735,5,0


In [38]:
# ---- schema checks for edited CSVs ----
required_order_products_cols = {'order_id', 'product_id', 'reordered','add_to_cart_order'}
required_orders_cols = {'order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order'}

missing_op_cols = required_order_products_cols - set(order_products_prior.columns)
missing_orders_cols = required_orders_cols - set(orders.columns)

if missing_op_cols:
    raise ValueError(f"order_products_final.csv missing columns: {sorted(missing_op_cols)}")

if missing_orders_cols:
    raise ValueError(f"orders_final.csv missing columns: {sorted(missing_orders_cols)}")

# keep only columns needed by this notebook so renamed extra columns (like id) do not break flow
order_products_prior = order_products_prior[['order_id', 'product_id', 'add_to_cart_order', 'reordered']].copy()

In [39]:
#for total_orders

total_orders = orders.groupby(['user_id'])['order_number'].max()
total_orders.head()

user_id
1    17
2    18
3     9
4    17
5    17
Name: order_number, dtype: int64

In [40]:
#this is for: reorder_ratio, dept_ratio  (1 row is 1 user)

departments_product = pd.merge(departments,products,on ='department_id',how='right')
departments_product.head()

departments_product_prior = pd.merge(departments_product,order_products_prior,on='product_id',how='right')
departments_product_prior.head()

departments_product_prior_order = pd.merge(departments_product_prior,orders,on='order_id',how='right')
departments_product_prior_order.tail()

departments_product_prior_order.drop(columns=['department_id','product_name','aisle_id','order_id','add_to_cart_order','order_number','order_dow','order_hour_of_day','days_since_prior_order'])
departments_product_prior_order.head()

,department_id,department,product_id,product_name,aisle_id,order_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,7,beverages,31140,Seltzer Water Calorie-Free Black Cherry,115,1,1,0,1,1,5,7,NaN
1,16,dairy eggs,486,Lowfat Raspberry Kefir Probiotic Cultured Milk...,120,1,2,0,1,1,5,7,NaN
2,16,dairy eggs,22325,Cocow Chocolate Milk,84,1,3,0,1,1,5,7,NaN
3,16,dairy eggs,12697,Sliced Pepper Jack Cheese,21,1,4,0,1,1,5,7,NaN
4,3,bakery,23735,Wheat Dinner Rolls,43,1,5,0,1,1,5,7,NaN


In [41]:
#this is for: evrything else  (1 row is 1 order)

per_order = pd.merge(orders,order_products_prior,on='order_id',how='left')
per_order.head()

per_order.drop(columns=['add_to_cart_order','product_id','reordered'])


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1,1,1,5,7,NaN
1,1,1,1,5,7,NaN
2,1,1,1,5,7,NaN
3,1,1,1,5,7,NaN
4,1,1,1,5,7,NaN
...,...,...,...,...,...,...
71364,8257,500,28,0,4,9.0
71365,8257,500,28,0,4,9.0
71366,8257,500,28,0,4,9.0
71367,8257,500,28,0,4,9.0


In [42]:
# for avg_products_per_order

products_per_order = per_order.groupby(['user_id','order_id'])['product_id'].count() 
# products_per_order.head()

avg_products_per_order = products_per_order.groupby(['user_id']).mean()
avg_products_per_order.head()

user_id
1    7.529412
2    7.666667
3    8.666667
4    7.411765
5    8.000000
Name: product_id, dtype: float64

In [43]:
#reorder_ratio

reorder_ratio= departments_product_prior_order.groupby(['user_id'])['reordered'].mean().reset_index()
reorder_ratio.head()

,user_id,reordered
0,1,0.500000
1,2,0.478261
2,3,0.474359
3,4,0.642857
4,5,0.529412


In [44]:
#avg_days_between_orders

avg_days_between_orders = orders.groupby(['user_id'])['days_since_prior_order'].mean()
avg_days_between_orders.head()

user_id
1    15.687500
2    14.529412
3    14.875000
4    13.812500
5    16.250000
Name: days_since_prior_order, dtype: float64

In [45]:
#weekend_ratio
orders.head()
orders['is_weekend'] = orders['order_dow'].isin([0,6])

weekend_ratio = orders.groupby('user_id')['is_weekend'].mean()
weekend_ratio

user_id
1      0.176471
2      0.111111
3      0.000000
4      0.294118
5      0.352941
         ...   
496    0.269231
497    0.296296
498    0.343750
499    0.285714
500    0.178571
Name: is_weekend, Length: 500, dtype: float64

In [46]:
#hour_of_day_distribution

orders.head()

orders['hour_distribution']= pd.cut(orders['order_hour_of_day'],bins = [-1, 5, 11, 17, 21, 23], labels =['night','morning','afternoon','evening','night'],ordered=False)
orders.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_weekend,hour_distribution
0,1,1,1,5,7,NaN,False,morning
1,2,1,2,6,23,11.0,True,night
2,3,1,3,2,8,9.0,False,morning
3,4,1,4,1,20,18.0,False,evening
4,5,1,5,3,11,12.0,False,morning


In [47]:
time = orders.groupby('user_id')['hour_distribution'].value_counts().unstack(fill_value=0)
time.tail()

time = time.div(time.sum(axis=1),axis=0)
time = time.reset_index() 
time.head()

time.iloc[:,0]

0        1
1        2
2        3
3        4
4        5
      ... 
495    496
496    497
497    498
498    499
499    500
Name: user_id, Length: 500, dtype: int64

In [48]:
# #dept_produce_ratio

total_vals_count = departments_product_prior_order['user_id'].value_counts()

In [49]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

encoded_vals = encoder.fit_transform(departments_product_prior_order[['department']])

new_cols = encoder.get_feature_names_out(['department'])

departments_ratio = pd.DataFrame(encoded_vals,columns=new_cols)
departments_ratio.head()

departments_ratio_finalboss = pd.concat(
    [departments_ratio, departments_product_prior_order['user_id'].reset_index(drop=True)],
    axis=1
)
departments_ratio_finalboss.head()


,department_bakery,department_beverages,department_breakfast,department_canned goods,department_dairy eggs,department_deli,department_dry goods pasta,department_frozen,department_household,department_meat seafood,department_other,department_pantry,department_personal care,department_produce,department_snacks,user_id
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [50]:
final_sum = departments_ratio_finalboss.groupby('user_id').sum()
final_sum = final_sum.drop(columns='department_nan', errors='ignore')

In [51]:
#final ratio 

dept_produce_ratio = final_sum.div(total_vals_count,axis=0)
dept_produce_ratio.head()

,department_bakery,department_beverages,department_breakfast,department_canned goods,department_dairy eggs,department_deli,department_dry goods pasta,department_frozen,department_household,department_meat seafood,department_other,department_pantry,department_personal care,department_produce,department_snacks
user_id,,,,,,,,,,,,,,,
1,0.039062,0.226562,0.0,0.0,0.359375,0.085938,0.0,0.062500,0.0,0.0,0.0,0.046875,0.0,0.164062,0.015625
2,0.028986,0.101449,0.0,0.0,0.289855,0.028986,0.0,0.086957,0.0,0.0,0.0,0.195652,0.0,0.123188,0.144928
3,0.025641,0.051282,0.0,0.0,0.192308,0.076923,0.0,0.089744,0.0,0.0,0.0,0.153846,0.0,0.217949,0.192308
4,0.000000,0.055556,0.0,0.0,0.396825,0.055556,0.0,0.023810,0.0,0.0,0.0,0.158730,0.0,0.134921,0.174603
5,0.029412,0.132353,0.0,0.0,0.485294,0.022059,0.0,0.029412,0.0,0.0,0.0,0.066176,0.0,0.066176,0.169118


In [52]:
dept_produce_ratio = dept_produce_ratio.drop(columns='department_nan', errors='ignore')
dept_produce_ratio.head()

,department_bakery,department_beverages,department_breakfast,department_canned goods,department_dairy eggs,department_deli,department_dry goods pasta,department_frozen,department_household,department_meat seafood,department_other,department_pantry,department_personal care,department_produce,department_snacks
user_id,,,,,,,,,,,,,,,
1,0.039062,0.226562,0.0,0.0,0.359375,0.085938,0.0,0.062500,0.0,0.0,0.0,0.046875,0.0,0.164062,0.015625
2,0.028986,0.101449,0.0,0.0,0.289855,0.028986,0.0,0.086957,0.0,0.0,0.0,0.195652,0.0,0.123188,0.144928
3,0.025641,0.051282,0.0,0.0,0.192308,0.076923,0.0,0.089744,0.0,0.0,0.0,0.153846,0.0,0.217949,0.192308
4,0.000000,0.055556,0.0,0.0,0.396825,0.055556,0.0,0.023810,0.0,0.0,0.0,0.158730,0.0,0.134921,0.174603
5,0.029412,0.132353,0.0,0.0,0.485294,0.022059,0.0,0.029412,0.0,0.0,0.0,0.066176,0.0,0.066176,0.169118


In [53]:
# unique_departments_count

unique_departments_count = final_sum.copy()
unique_departments_count = unique_departments_count.drop(columns='department_nan', errors='ignore')
unique_departments_count.head()

,department_bakery,department_beverages,department_breakfast,department_canned goods,department_dairy eggs,department_deli,department_dry goods pasta,department_frozen,department_household,department_meat seafood,department_other,department_pantry,department_personal care,department_produce,department_snacks
user_id,,,,,,,,,,,,,,,
1,5.0,29.0,0.0,0.0,46.0,11.0,0.0,8.0,0.0,0.0,0.0,6.0,0.0,21.0,2.0
2,4.0,14.0,0.0,0.0,40.0,4.0,0.0,12.0,0.0,0.0,0.0,27.0,0.0,17.0,20.0
3,2.0,4.0,0.0,0.0,15.0,6.0,0.0,7.0,0.0,0.0,0.0,12.0,0.0,17.0,15.0
4,0.0,7.0,0.0,0.0,50.0,7.0,0.0,3.0,0.0,0.0,0.0,20.0,0.0,17.0,22.0
5,4.0,18.0,0.0,0.0,66.0,3.0,0.0,4.0,0.0,0.0,0.0,9.0,0.0,9.0,23.0


In [54]:
#top_dept_concentration

dept_produce_ratio['concentration'] = dept_produce_ratio.max(axis=1)
dept_produce_ratio.head()

top_dept_concentration = dept_produce_ratio[['concentration']]
top_dept_concentration.head()

,concentration
user_id,
1,0.359375
2,0.289855
3,0.217949
4,0.396825
5,0.485294


In [55]:
# now we will merge all feature blocks for user-user similarity

join1 = pd.merge(total_orders, avg_products_per_order, on='user_id')
join1.rename(columns={'order_number': 'total_orders', 'product_id': 'avg_products_per_order'}, inplace=True)

join2 = pd.merge(join1, reorder_ratio, on='user_id')
join2.rename(columns={'reordered': 'reorder_ratio'}, inplace=True)

join3 = pd.merge(join2, avg_days_between_orders, on='user_id')
join3.rename(columns={'days_since_prior_order': 'avg_days_between_orders'}, inplace=True)

# normalize department feature columns in case _x/_y duplicates were introduced earlier
dept_features = dept_produce_ratio.reset_index().copy()
if 'index' in dept_features.columns and 'user_id' not in dept_features.columns:
    dept_features = dept_features.rename(columns={'index': 'user_id'})

selected_cols = ['user_id']
seen = set()
for col in dept_features.columns:
    if col == 'user_id':
        continue
    base_col = col.removesuffix('_x').removesuffix('_y')
    if base_col == 'department_nan':
        continue
    if base_col in seen:
        continue
    seen.add(base_col)
    selected_cols.append(col)

dept_features = dept_features[selected_cols].copy()
dept_features = dept_features.rename(columns={c: c.removesuffix('_x').removesuffix('_y') for c in dept_features.columns})

join4 = pd.merge(join3, dept_features, on='user_id', how='inner')
join5 = pd.merge(join4, time, on='user_id', how='inner')
df = pd.merge(join5, weekend_ratio.reset_index(name='weekend_ratio'), on='user_id', how='inner')

# clean feature matrix so scaling/normalize does not fail on NaN/inf
feature_cols = [c for c in df.columns if c != 'user_id']
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

df.head()


,user_id,total_orders,avg_products_per_order,reorder_ratio,avg_days_between_orders,department_bakery,department_beverages,department_breakfast,department_canned goods,department_dairy eggs,...,department_pantry,department_personal care,department_produce,department_snacks,concentration,afternoon,evening,morning,night,weekend_ratio
0,1,17,7.529412,0.500000,15.687500,0.039062,0.226562,0.0,0.0,0.359375,...,0.046875,0.0,0.164062,0.015625,0.359375,0.235294,0.117647,0.588235,0.058824,0.176471
1,2,18,7.666667,0.478261,14.529412,0.028986,0.101449,0.0,0.0,0.289855,...,0.195652,0.0,0.123188,0.144928,0.289855,0.388889,0.000000,0.555556,0.055556,0.111111
2,3,9,8.666667,0.474359,14.875000,0.025641,0.051282,0.0,0.0,0.192308,...,0.153846,0.0,0.217949,0.192308,0.217949,0.111111,0.111111,0.555556,0.222222,0.000000
3,4,17,7.411765,0.642857,13.812500,0.000000,0.055556,0.0,0.0,0.396825,...,0.158730,0.0,0.134921,0.174603,0.396825,0.411765,0.117647,0.411765,0.058824,0.294118
4,5,17,8.000000,0.529412,16.250000,0.029412,0.132353,0.0,0.0,0.485294,...,0.066176,0.0,0.066176,0.169118,0.485294,0.294118,0.117647,0.529412,0.058824,0.352941


In [56]:
# now similarity and vector embeddings

user_ids = df['user_id'].astype(int)
X = df.drop(columns=['user_id']).copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
scaled_df = scale.fit_transform(X)

scaled_df = pd.DataFrame(
    scaled_df,
    columns=X.columns,
    index=user_ids
)

scaled_df.head()


,total_orders,avg_products_per_order,reorder_ratio,avg_days_between_orders,department_bakery,department_beverages,department_breakfast,department_canned goods,department_dairy eggs,department_deli,...,department_pantry,department_personal care,department_produce,department_snacks,concentration,afternoon,evening,morning,night,weekend_ratio
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.064528,-0.506341,0.616281,0.332743,-0.166203,1.498700,-0.704234,-0.460817,1.348785,1.079553,...,-1.204406,-0.454131,-0.041071,-0.910062,0.340928,-0.868379,-0.647671,1.330513,-0.049827,-0.882752
2,0.197301,-0.475602,0.459868,0.097737,-0.421299,-0.285795,-0.704234,-0.460817,0.802368,-0.233832,...,2.159430,-0.454131,-0.337961,0.023506,-0.245001,0.182981,-1.527908,1.130731,-0.088464,-1.376656
3,-0.997656,-0.251648,0.431794,0.167866,-0.505964,-1.001332,-0.704234,-0.460817,0.035659,0.871669,...,1.214201,-0.454131,0.350333,0.365592,-0.851043,-1.718414,-0.696573,1.130731,1.882012,-2.216293
4,0.064528,-0.532688,1.644141,-0.047743,-1.155058,-0.940379,-0.704234,-0.460817,1.643141,0.378907,...,1.324627,-0.454131,-0.252743,0.237765,0.656569,0.339566,-0.647671,0.251689,-0.049827,0.006275
5,0.064528,-0.400951,0.827900,0.446888,-0.410509,0.154986,-0.704234,-0.460817,2.338493,-0.393570,...,-0.768002,-0.454131,-0.752068,0.198160,1.402202,-0.465730,-0.647671,0.970905,-0.049827,0.450788


In [57]:
#now we will assign weights 

behavior_cols = [
    'total_orders',
    'avg_products_per_order',
    'reorder_ratio',
    'avg_days_between_orders'
]

dept_ratio_cols = [c for c in scaled_df.columns  if c.startswith('department_')]


WEIGHT_BEHAVIOR = 1.0
WEIGHT_DEPT_RATIO = 3.0

weighted_df = scaled_df.copy()

weighted_df[behavior_cols] *= WEIGHT_BEHAVIOR
weighted_df[dept_ratio_cols] *= WEIGHT_DEPT_RATIO


In [58]:
from sklearn.preprocessing import normalize

weighted_df = weighted_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_emb = normalize(weighted_df)


In [66]:
from sklearn.neighbors import NearestNeighbors

n_neighbors = min(10, len(weighted_df))
knn = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine')
knn.fit(X_emb)

query_user_id = int(user_ids.iloc[2])
query_row_idx = weighted_df.index.get_loc(query_user_id)

distances, indices = knn.kneighbors([X_emb[query_row_idx]])

similar_user_ids = weighted_df.index[indices[0]].tolist()

print(f"Query user_id: {query_user_id}")
print("Top similar user_ids:", similar_user_ids)
print("Distances:", distances[0])


Query user_id: 3
Top similar user_ids: [3, 37, 16, 2, 8, 67, 459, 64, 93, 4]
Distances: [0.         0.24125868 0.2414756  0.27710484 0.3111695  0.37948841
 0.40902166 0.41939547 0.42313793 0.43443177]


In [60]:
import joblib
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Collect all objects
user_recommender_data = {
    'knn': knn,
    'X_emb': X_emb,  # Normalized embeddings
    'user_id_list': user_ids.tolist(),  # User IDs in order
}

# Save with compression
joblib.dump(
    user_recommender_data,
    'backend/models/user_recomm.joblib',
    compress=3
)

print("✓ User recommender model saved!")
print(f"  Location: backend/models/user_recommender_data_final.joblib")
file_size = os.path.getsize('backend/models/user_recomm.joblib') / (1024**2)
print(f"  Total users: {len(user_ids)}")

✓ User recommender model saved!
  Location: backend/models/user_recommender_data_final.joblib
  Total users: 500
